<div dir="rtl">

# 🦀 04 - Qdrant Vector Store in LangChain

## ما هو Qdrant؟
- **Qdrant** هو محرك بحث متجهي فائق الأداء والسرعة مكتوب بلغة **Rust**، مصمم للبيئات الإنتاجية والضخمة (Production-Ready).
- يتميز بنظام **Payload Filtering** المتقدم جداً الذي يدمج بين البحث المتجهي والفلترة الهيكلية بدقة فائقة.

---

### 🌟 أهم مميزات Qdrant:
1. **أداء استثنائي بلغة Rust**: سرعة معالجة عالية جداً واستهلاك ذاكرة منخفض.
2. **أنماط تشغيل مرنة**:
   - وضع الذاكرة (`location=":memory:"`) للتجارب السريعة.
   - التخزين على القرص (`path="./data/qdrant_db"`).
   - وضع الخادم أو السحابة (`url="http://localhost:6333"` أو Qdrant Cloud).
3. **دعم التكميم (Quantization)**: تقليص حجم المتجهات لتوفير الذاكرة مع الحفاظ على دقة البحث.

</div>


<div dir="rtl">

### 1️⃣ تجهيز المكتبات ونموذج التضمين

</div>


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ إنشاء Qdrant في الذاكرة (In-Memory Mode) وإضافة المستندات

</div>


In [ ]:
docs = [
    Document(
        page_content="تعلم لغة Rust يوفر أداءً يضاهي C++ مع ضمانات أمان الذاكرة التامة بدون Garbage Collector.",
        metadata={"lang": "Rust", "type": "Systems", "difficulty": "Advanced"}
    ),
    Document(
        page_content="تعتبر لغة Python الخيار الأول للذكاء الاصطناعي وبناء خطوط أنابيب RAG ونماذج اللغة.",
        metadata={"lang": "Python", "type": "AI", "difficulty": "Beginner"}
    ),
    Document(
        page_content="لغة Go تتميز بالبساطة العالية ودعم Concurrency القوي لبناء الخوادم المصغرة والـ Microservices.",
        metadata={"lang": "Go", "type": "Backend", "difficulty": "Intermediate"}
    ),
    Document(
        page_content="تقنيات TypeScript تضيف نظام الأنواع الثابتة إلى JavaScript مما يقلل الأخطاء البرمجية في المشاريع الكبيرة.",
        metadata={"lang": "TypeScript", "type": "Frontend", "difficulty": "Intermediate"}
    )
]

# إنشاء مستودع Qdrant في الذاكرة
qdrant_store = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="programming_languages"
)

print("✅ تم إنشاء Qdrant VectorStore وإضافة المستندات بنجاح!")


<div dir="rtl">

### 3️⃣ البحث الدلالي مع حساب درجة التطابق (Similarity Search with Score)

</div>


In [ ]:
query = "ما هي اللغة الأفضل لبناء نماذج الذكاء الاصطناعي والـ RAG؟"

results = qdrant_store.similarity_search_with_score(query, k=2)

print(f"🔍 الاستعلام: {query}\n")
for doc, score in results:
    print(f"درجة التطابق (Cosine Similarity Score): {score:.4f}")
    print(f"المحتوى: {doc.page_content}")
    print(f"الميتاداتا: {doc.metadata}")
    print("-" * 50)


<div dir="rtl">

### 4️⃣ البحث مع الفلترة بالـ Payload / Metadata
تطبيق شروط فلترة مباشرة على بيانات الـ Payload المصاحبة لكل متجه.

</div>


In [ ]:
# البحث مع اشتراط الفئة Systems
results_filtered = qdrant_store.similarity_search(
    "لغات برمجة لبناء الخوادم والأنظمة السريعة",
    k=2,
    filter={"type": "Systems"}
)

print("🎯 نتائج الفلترة (Systems فقط):")
for doc in results_filtered:
    print(f"• {doc.page_content} ({doc.metadata})")


<div dir="rtl">

### 5️⃣ حفظ مستودع Qdrant على القرص المحلي (Local Storage Path)
بدلاً من تشغيله في الذاكرة فقط، يمكن تمرير مسار مجلد محلي ليتم حفظ الفهارس واسترجاعها تلقائياً.

</div>


In [ ]:
qdrant_disk_path = "../../data/qdrant_db"
os.makedirs(qdrant_disk_path, exist_ok=True)

# إنشاء مستودع محفوظ على القرص
qdrant_disk_store = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    path=qdrant_disk_path,
    collection_name="disk_collection"
)

print(f"💾 تم حفظ مستودع Qdrant على القرص في: {os.path.abspath(qdrant_disk_path)}")
